# Environment smoke episode
Run a seeded episode with the same features, cash return, and cost conventions used by training.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, yaml
from src.data.cleaner import align_common_dates
from src.features.pipeline import build_feature_splits
from src.environment.portfolio_env import PortfolioEnv
root=Path.cwd(); cfg=yaml.safe_load((root/'configs/config.yaml').read_text())
prices=pd.concat([pd.read_csv(root/f'data/processed/{s}_prices.csv',index_col='date',parse_dates=['date']) for s in ['train','validation','test']]).sort_index()
prices,_=align_common_dates(prices)
features,_=build_feature_splits(prices,train_end=cfg['project']['train']['end'],validation_end=cfg['project']['validation']['end'],test_start=cfg['project']['test']['start'])
panel=features['train']; returns=prices.pct_change(fill_method=None); returns['CASH']=(1+cfg['portfolio']['synthetic_cash_annual_return'])**(1/cfg['portfolio']['cash_day_count'])-1; returns=returns.reindex(panel.index)
env=PortfolioEnv(panel,returns,transaction_cost_rate=cfg['portfolio']['transaction_cost_rate']); obs,info=env.reset(seed=42)
done=False; rewards=[]
while not done:
    obs,reward,terminated,truncated,info=env.step(env.action_space.sample()); rewards.append(reward); done=terminated or truncated
info